# EECE 5554 Lab 3: VectorNav IMU Driver and Noise Characterization

## Learning objectives

Lab 3 introduces the inertial measurement unit (IMU). You will write a ROS 2 driver for a VectorNav VN-100 that configures the sensor over its serial port and publishes its gyroscope, accelerometer, magnetometer, and orientation data in standard ROS 2 messages. Then you will measure how noisy the gyroscope is, with Allan variance, and work out what that noise means for a robot that relies on it.

By the end of this lab a well-prepared student will be able to:
<ul>
    <li> Configure a sensor by writing its registers over a serial port, and confirm the change took effect </li>
    <li> Validate a checksummed sentence, and keep a driver running when the input is damaged </li>
    <li> Convert sensor output to ROS 2 conventions: SI units, quaternions, and frame IDs </li>
    <li> Publish a custom message that nests sensor_msgs/Imu and sensor_msgs/MagneticField </li>
    <li> Compute an Allan deviation and read angle random walk, bias instability, and rate random walk from it </li>
    <li> Predict the heading error of a robot that integrates its gyro, from those three numbers </li>
</ul>

## Prerequisites

| Item | Explain |
|---|---|
| Lab 1 | Complete and working. You reuse the serial reading, the two-package pattern, the launch file, and rosbag recording |
| PA0, PA1 | Ubuntu 24.04 and ROS 2 Jazzy, as before |
| Hardware | VectorNav VN-100 Rugged with its USB cable, issued in the classroom and returned after your recordings |

## Software to install

```bash
sudo apt install minicom python3-serial python3-numpy python3-matplotlib python3-ipykernel
```

You do not need AllanTools. Part 3 of this notebook includes a short NumPy implementation of the Allan deviation that handles five hours of data in about a second.

Make sure the course repository is current. The IMU emulator, the self-check, and the test vectors are in `~/5554-F26/sensor_emulator`:

```bash
cd ~/5554-F26 && git pull
```

### How to run this notebook

Copy it into your `Lab3/analysis/` folder and use it as the start of your analysis. The analysis code reads rosbags, so open it from a terminal where ROS 2 and your workspace are sourced, and pick the system Python 3.12 as the kernel:

```bash
source /opt/ros/jazzy/setup.bash
source ~/EECE5554/Lab3/ros2_ws/install/setup.bash
code ~/EECE5554/Lab3/analysis
```

A virtual environment hides `rclpy` and `rosbag2_py`, so do not use one for this lab.

## Individual and team work

**Individual:** write your own driver, record your own stationary data set, run your own analysis, write your own answers.
**Shared:** the VN-100 units, and the motion data set, which you record with a teammate. Sharing the motion rosbag is expected. Everything downstream of the bags is yours.

## Collaboration statement

You should complete this assignment without collaboration with other students in the course or elsewhere, beyond the shared hardware and shared motion data noted above. Please ask for help from the TAs or course instructors if you are getting stuck.

## Additional help

A complete worked example, with the message definition, a driver skeleton, the launch file, and a sample passing report, is in `EECE5554_Lab3_example.md`. The Lab 1 help notebooks in `Lab1/help/` still apply for ROS 2 basics and rosbags.

# Part 1: The driver

## The VN-100 in one page

The VN-100 streams one ASCII sentence per sample. In this lab it streams `$VNYMR`: yaw, pitch, roll, magnetic, acceleration, and rate.

```
$VNYMR,-032.475,+001.828,-000.917,+00.1474,+00.0965,+00.4968,+00.332,+00.140,-09.773,-00.001393,+00.000136,-00.000320*6C
```

| Fields | Meaning | VN-100 unit | What ROS 2 expects |
|---|---|---|---|
| 1 to 3 | Yaw, pitch, roll | degrees | a quaternion in `imu.orientation` |
| 4 to 6 | Magnetic field x, y, z | gauss | tesla, in `mag_field.magnetic_field` |
| 7 to 9 | Acceleration x, y, z | m/s<sup>2</sup> | m/s<sup>2</sup>, in `imu.linear_acceleration` |
| 10 to 12 | Angular rate x, y, z | rad/s | rad/s, in `imu.angular_velocity` |
| after `*` | Checksum | two hex digits | not published, but you must check it |

Four facts you will need:
<ul>
    <li> Yaw, pitch, roll are a 3-2-1 sequence: yaw about z, then pitch about the new y, then roll about the new x </li>
    <li> The VN-100 body frame is x forward, y right, z down. ROS (REP 103) uses x forward, y left, z up. You publish in the sensor's own frame and say so with the frame_id <code>imu1_frame</code>. Question 1 comes back to this </li>
    <li> The angular rates in <code>$VNYMR</code> are already corrected by the onboard filter's estimate of the gyro bias. Question 3 comes back to this </li>
    <li> The serial port runs at 115200 baud </li>
</ul>

### The checksum

The checksum is the XOR of every character between `$` and `*`, printed as two upper case hex digits. It is the same algorithm as the NMEA checksum from Lab 1. The device uses it on what it sends, and it checks it on what you send. Run the cell below.

In [ ]:
def vn_checksum(body):
    """XOR of every character between $ and *, as two upper case hex digits."""
    x = 0
    for ch in body.encode("ascii"):
        x ^= ch
    return f"{x:02X}"


line = ("$VNYMR,-032.475,+001.828,-000.917,+00.1474,+00.0965,+00.4968,"
        "+00.332,+00.140,-09.773,-00.001393,+00.000136,-00.000320*6C")
body, cs = line[1:].split("*")
print("checksum in the sentence:", cs, "  computed:", vn_checksum(body))
print("command that sets 40 Hz :", f"$VNWRG,07,40*{vn_checksum('VNWRG,07,40')}")

### Registers: how you configure the device

The VN-100 is configured by writing registers. Two matter in this lab.

| Register | Name | Value you want | Command |
|---|---|---|---|
| 6 | Asynchronous data output type | 14, which is `$VNYMR` | `$VNWRG,06,14*59` |
| 7 | Asynchronous data output frequency | 40 Hz | `$VNWRG,07,40*59` |

The allowed rates are 1, 2, 4, 5, 10, 20, 25, 40, 50, 100, and 200 Hz.

<ul>
    <li> Every command ends with <code>\r\n</code>. Without it the device keeps waiting and does nothing </li>
    <li> pyserial sends bytes, not strings: <code>ser.write(command.encode("ascii"))</code> </li>
    <li> <code>XX</code> in place of the checksum skips the check, which is handy at a terminal. In your driver, compute the real one </li>
    <li> A successful write is echoed back into the data stream: <code>$VNWRG,07,40*59</code>. A failed one gets <code>$VNERR,</code> and a code </li>
    <li> You can read a register back: <code>$VNRRG,07*XX</code> answers <code>$VNRRG,07,40*5C</code> </li>
    <li> Writes are lost at power off unless saved with <code>$VNWNV</code>. Do not save settings on the shared units. Set the rate every time your driver starts instead </li>
</ul>

| Error code | Meaning |
|---|---|
| 03 | Invalid checksum |
| 04 | Invalid command |
| 05 | Not enough parameters |
| 06 | Too many parameters |
| 07 | Invalid parameter, for example a rate that is not allowed |
| 08 | Invalid register |

**Do not get 40 Hz by sleeping or throttling in your driver.** The self-check reads the emulator's log to confirm that register 7 was written, and the emulator starts at 10 Hz, so a driver cannot reach 40 Hz any other way.

## Step 1: Create the two packages

Same pattern as Lab 1: a CMake package that defines the message, and a Python package that holds the driver. A Python package cannot generate messages, which is the only reason there are two.

```bash
mkdir -p ~/EECE5554/Lab3/ros2_ws/src
cd ~/EECE5554/Lab3/ros2_ws/src

ros2 pkg create vn_msgs --build-type ament_cmake
ros2 pkg create vn_driver --build-type ament_python --dependencies rclpy std_msgs sensor_msgs vn_msgs
```

Target layout:

```
Lab3/ros2_ws/src/
├── vn_msgs/
│   ├── CMakeLists.txt
│   ├── package.xml
│   └── msg/
│       └── Vectornav.msg
└── vn_driver/
    ├── vn_driver/
    │   ├── __init__.py
    │   └── driver.py
    ├── launch/
    │   └── driver.launch.py
    ├── resource/
    │   └── vn_driver
    ├── package.xml
    ├── setup.cfg
    └── setup.py
```

**Names matter.** The self-check and the grader look for exactly `vn_msgs`, `vn_driver`, `Vectornav.msg`, `driver.py`, and `driver.launch.py`. Capitalization counts.

## Step 2: Define the message

Create `vn_msgs/msg/Vectornav.msg`:

```
std_msgs/Header header
sensor_msgs/Imu imu
sensor_msgs/MagneticField mag_field
string vnymr_read
```

| Field | What goes in it |
|---|---|
| `header` | stamp from the system clock, nanoseconds included; frame_id `imu1_frame` |
| `imu` | `orientation` as a quaternion, `angular_velocity` in rad/s, `linear_acceleration` in m/s<sup>2</sup>. Fill `imu.header` the same as `header` |
| `mag_field` | `magnetic_field` in tesla. Fill `mag_field.header` the same as `header` |
| `vnymr_read` | the whole sentence as received, from `$` to the checksum, without `\r\n` |

Leave the covariance arrays at zero, which ROS reads as "unknown".

In `vn_msgs/CMakeLists.txt`, add these before `ament_package()`, exactly as you did for `Customgps` in Lab 1, with `sensor_msgs` added:

```cmake
find_package(rosidl_default_generators REQUIRED)
find_package(std_msgs REQUIRED)
find_package(sensor_msgs REQUIRED)

rosidl_generate_interfaces(${PROJECT_NAME}
  "msg/Vectornav.msg"
  DEPENDENCIES std_msgs sensor_msgs
)
```

`vn_msgs/package.xml` needs `rosidl_default_generators` as a buildtool dependency, `std_msgs` and `sensor_msgs` as dependencies, `rosidl_default_runtime` as an exec dependency, and `<member_of_group>rosidl_interface_packages</member_of_group>`.

**Checkpoint.**

```bash
cd ~/EECE5554/Lab3/ros2_ws
colcon build --symlink-install
source install/setup.bash
ros2 interface show vn_msgs/msg/Vectornav
```

If the last command does not print your four fields, nothing downstream will work. Fix it now.

## Step 3: Write the driver

Create `vn_driver/vn_driver/driver.py`. It must:

1. Take `port` (default `/dev/ttyUSB0`) and `baudrate` (default `115200`) as ROS 2 parameters
2. Open the port with pyserial
3. Write register 6 with 14 and register 7 with 40, with computed checksums and `\r\n`. Watch the stream for the echo or a `$VNERR`, and log which one you got
4. Then loop: read a line, validate it, convert it, publish it as `Vectornav` on `/imu`

**Validation.** A line is good only if it starts with `$VNYMR,`, its checksum matches, it has exactly 12 fields after the header, and every field converts to a number. Skip anything else and keep going. One bad line must never stop the node.

**Conversions.**
<ul>
    <li> Magnetic field: gauss times 1e-4 gives tesla </li>
    <li> Orientation: degrees to radians, then to a quaternion with the formula below </li>
    <li> Angular rate and acceleration: already in SI units, publish them as they are </li>
</ul>

With $\psi$ yaw, $\theta$ pitch, $\phi$ roll in radians, and $c_\psi = \cos(\psi/2)$, $s_\psi = \sin(\psi/2)$, and so on:

$$q_x = s_\phi c_\theta c_\psi - c_\phi s_\theta s_\psi \qquad q_y = c_\phi s_\theta c_\psi + s_\phi c_\theta s_\psi$$

$$q_z = c_\phi c_\theta s_\psi - s_\phi s_\theta c_\psi \qquad q_w = c_\phi c_\theta c_\psi + s_\phi s_\theta s_\psi$$

You may use a library instead, such as `scipy.spatial.transform.Rotation`, but check whichever you use against this table. A quaternion and its negative describe the same rotation, and the self-check accepts either.

| yaw | pitch | roll | x | y | z | w |
|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0.000000 | 0.000000 | 0.000000 | 1.000000 |
| 90 | 0 | 0 | 0.000000 | 0.000000 | 0.707107 | 0.707107 |
| 0 | 0 | 90 | 0.707107 | 0.000000 | 0.000000 | 0.707107 |
| -32.475 | 1.828 | -0.917 | -0.003222 | 0.017552 | -0.279452 | 0.959994 |
| 150 | 45 | -60 | -0.439680 | -0.360423 | 0.822363 | 0.022260 |

**Header.** Use `self.get_clock().now().to_msg()` and copy the same stamp and frame_id into `header`, `imu.header`, and `mag_field.header`. Note what this stamp means: when the line arrived at your laptop, not when the VN-100 sampled it.

### The trap that costs somebody their data every year

Two kinds of line arrive that are not good `$VNYMR` sentences, and each one kills a driver that does not expect it.

**The echo.** After you write register 7, the device answers `$VNWRG,07,40*59` in the middle of the data. It is not a `$VNYMR` sentence. Parse it as one and `float("40*59")` raises `ValueError`.

**Damaged lines.** Real serial data gets damaged: a sentence cut short because you opened the port in the middle of it, two sentences run together, a flipped digit that only the checksum catches, an empty field, bytes that are not text at all. `raw.decode("utf-8")` raises `UnicodeDecodeError` on the last one, and `float("")` raises `ValueError` on an empty field. Decode with `raw.decode("ascii", errors="replace")`, validate before you convert, and catch `ValueError`.

On the hardware, a crash like this happens three minutes into your stationary recording, and you start over. The self-check's third test vector, `imu_corrupt.txt`, contains every kind of damage above, so you find out now instead.

## Step 4: The launch file

Create `vn_driver/launch/driver.launch.py`. It declares a `port` argument and passes it to the node as the `port` parameter. Install it through `data_files` in `setup.py`, as in Lab 1, and add `driver = vn_driver.driver:main` to `console_scripts`. The worked example has a complete one.

```bash
ros2 launch vn_driver driver.launch.py port:=/dev/ttyUSB0
```

## Step 5: Test against the emulator

The IMU emulator behaves like a VN-100 on a virtual serial port. It streams a test vector, and it answers your register commands the way the device does.

```bash
cd ~/5554-F26/sensor_emulator
python3 imu_emulator.py -f data/imu_stationary.txt
```

It prints `Pseudo serial port: /dev/pts/N`. In a second terminal, with your workspace sourced:

```bash
ros2 launch vn_driver driver.launch.py port:=/dev/pts/N
```

In a third, `ros2 topic hz /imu` should settle at 40 once your register write works, and `ros2 topic echo /imu --once` shows one message.

The emulator starts at 10 Hz on purpose. Lab units are shared, and a unit can come up at whatever rate the last user saved, so your driver must set the rate itself. The emulator's terminal shows every command it receives and its answer:

```
[   2.102 s] received $VNWRG,07,40*59
[   2.102 s]   OK: register 7 (output rate) is now 40 Hz
[   2.102 s]   replied $VNWRG,07,40*59
```

| Test vector | What it checks |
|---|---|
| `data/imu_stationary.txt` | units, headers, the raw sentence, and the output rate |
| `data/imu_rotation.txt` | the quaternion, through full turns in yaw and roll and large swings in pitch |
| `data/imu_corrupt.txt` | that your driver survives damaged lines and publishes none of them |

## Step 6: Run the self-check

```bash
cd ~/5554-F26/sensor_emulator
./verify_lab3.sh <your-northeastern-username>
```

It builds your workspace in a temporary folder, runs your driver against all three test vectors through `ros2 launch` with a different port each time, checks your rosbags and your Git hygiene, and writes `~/EECE5554/Lab3/[NortheasternUsername]_lab3.txt`. Every failed check says why. Fix what fails and run it again.

Get the driver checks passing before you take a VN-100, because the units are shared. Run it once more after Step 8 so the report includes your rosbags, complete Section 9 of the report, and commit it as generated.

## Step 7: Connect the VN-100

1. Plug it in and run `dmesg | tail -20`. You should see a new `ttyUSB0` and the USB serial converter's driver, usually `ftdi_sio`.

2. Look at the raw stream, then close minicom before you start your driver:

    ```bash
    minicom -D /dev/ttyUSB0 -b 115200
    ```

3. Lower the USB latency. An FTDI converter holds bytes for up to 16 ms by default and delivers them in bursts, which shows up as jitter in your time stamps. If `dmesg` showed `ftdi_sio`:

    ```bash
    echo 'ACTION=="add", SUBSYSTEM=="usb-serial", DRIVER=="ftdi_sio", ATTR{latency_timer}="1"' \
        | sudo tee /etc/udev/rules.d/49-usb-latency.rules
    sudo udevadm control --reload-rules && sudo udevadm trigger
    ```

    Unplug, plug back in, and confirm that `cat /sys/bus/usb-serial/devices/ttyUSB0/latency_timer` prints `1`. Check it each time you record. The self-check prints it too.

4. Permissions are already handled: you joined the `dialout` group in PA0. You do not need a udev permission rule or `chmod`.

5. Run your driver on the real port and confirm the rate:

    ```bash
    ros2 launch vn_driver driver.launch.py port:=/dev/ttyUSB0
    ros2 topic hz /imu
    ```

## Step 8: Record your two data sets

### Stationary data set (individual, at least 5 minutes)

Put the VN-100 on a solid surface, away from computers, monitors, phones, motors, and people walking past. Not on your laptop, and not on a desk you are typing on. Do not touch it or its cable while it records.

```bash
mkdir -p ~/EECE5554/Lab3/data && cd ~/EECE5554/Lab3/data
ros2 bag record -o stationary /imu
```

Let it run for at least 5 minutes, then stop it with Ctrl+C. Aim for 6 so you have margin.

### Motion data set (with a teammate, 1 to 2 minutes)

Work through this sequence with the unit on a table, and note when each segment starts. A phone video helps you find the segments later. Do not commit the video.

| Segment | Motion | About |
|---|---|---|
| 1 | Still, flat on the table | 10 s |
| 2 | Turn it about 90 degrees counterclockwise, seen from above, hold, turn it back | 10 s |
| 3 | Tip it onto its side (roll about 90 degrees), hold, tip it back | 10 s |
| 4 | Tip it nose up (pitch about 90 degrees), hold, tip it back | 10 s |
| 5 | Tap the table next to it three times | 5 s |
| 6 | Shake it gently in your hand | 5 s |
| 7 | Still | 10 s |

```bash
cd ~/EECE5554/Lab3/data
ros2 bag record -o motion /imu
```

Be gentle with the units: no swinging them by the cable. Each teammate commits a copy of the motion bag. Check both bags with `ros2 bag info`, then rerun the self-check.

### The long data set (provided)

Allan variance needs hours of stationary data, so the course provides a five-hour recording from a VN-100: [DATA_LINK](https://northeastern-my.sharepoint.com/:u:/g/personal/x_li1_northeastern_edu/IQDVKWl4ApHsTaU5bbrGZherAafWf17OQwI6wi3nncHN7RE). It holds the raw sentences, including some damaged ones, so you parse it with your own driver's validation. It was recorded at `~39.5 Hz` Hz.

Save it **outside** your repository, for example in `~/lab3_long_data/`. It is too large for GitHub.

If your own recording fails, a provided stationary and motion set is at the same link. It protects your analysis marks but not your data set marks.

# Part 2: Analysis

You will produce six figures, a noise parameter table, and answers to five questions. The cells below give you the tools: reading your rosbags, and in Part 3, the Allan deviation. The plots for Figures 1 to 4 and 6 are yours to write.

Run the cells in order. Cells that need data you have not recorded yet print a reminder instead of failing.

In [ ]:
import gzip
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

### Reading your rosbags

`read_imu_bag` reads a rosbag2 folder of `Vectornav` messages into NumPy arrays. Time starts at 0 from the first header stamp.

In [ ]:
def read_imu_bag(path, topic="/imu"):
    """Read a rosbag2 folder of vn_msgs/Vectornav into NumPy arrays."""
    import yaml
    import rosbag2_py
    from rclpy.serialization import deserialize_message
    from rosidl_runtime_py.utilities import get_message

    path = Path(path).expanduser()
    meta = yaml.safe_load((path / "metadata.yaml").read_text())["rosbag2_bagfile_information"]
    reader = rosbag2_py.SequentialReader()
    reader.open(rosbag2_py.StorageOptions(uri=str(path), storage_id=meta["storage_identifier"]),
                rosbag2_py.ConverterOptions("cdr", "cdr"))
    types = {t.name: t.type for t in reader.get_all_topics_and_types()}
    msg_type = get_message(types[topic])
    reader.set_filter(rosbag2_py.StorageFilter(topics=[topic]))
    rows = []
    while reader.has_next():
        _, data, _ = reader.read_next()
        m = deserialize_message(data, msg_type)
        o, w = m.imu.orientation, m.imu.angular_velocity
        a, b = m.imu.linear_acceleration, m.mag_field.magnetic_field
        rows.append([m.header.stamp.sec + 1e-9 * m.header.stamp.nanosec,
                     o.x, o.y, o.z, o.w, w.x, w.y, w.z, a.x, a.y, a.z, b.x, b.y, b.z])
    r = np.array(rows)
    return {"t": r[:, 0] - r[0, 0], "quat": r[:, 1:5], "gyro": r[:, 5:8],
            "accel": r[:, 8:11], "mag": r[:, 11:14]}

In [ ]:
STATIONARY = Path("~/EECE5554/Lab3/data/stationary").expanduser()
MOTION = Path("~/EECE5554/Lab3/data/motion").expanduser()

stat = read_imu_bag(STATIONARY) if (STATIONARY / "metadata.yaml").exists() else None
motion = read_imu_bag(MOTION) if (MOTION / "metadata.yaml").exists() else None

for name, d in (("stationary", stat), ("motion", motion)):
    if d is None:
        print(f"{name}: not recorded yet (Step 8)")
    else:
        dt = np.diff(d["t"])
        print(f"{name}: {len(d['t'])} messages over {d['t'][-1]:.1f} s, "
              f"{1 / np.median(dt):.1f} Hz, largest gap {1000 * dt.max():.0f} ms")

### Back to Euler angles

Figure 3 and Question 5 need yaw, pitch, and roll, recovered from the quaternions you published. For a quaternion $(x, y, z, w)$ and the 3-2-1 sequence:

$$\psi = \operatorname{atan2}\big(2(wz + xy),\ 1 - 2(y^2 + z^2)\big) \qquad \theta = \arcsin\big(2(wy - zx)\big) \qquad \phi = \operatorname{atan2}\big(2(wx + yz),\ 1 - 2(x^2 + y^2)\big)$$

Clip the argument of $\arcsin$ to $[-1, 1]$ before you take it. Write this yourself, or use `Rotation.from_quat(q).as_euler("ZYX", degrees=True)` from `scipy.spatial.transform`. Either way, check a few samples against the numbers in `vnymr_read`. They must match to the printed precision. Yaw jumps from +180 to -180 as the unit turns through south, so use `np.unwrap` before you difference or integrate it.

### Required figures

Every figure needs axis labels with units, a legend where there is more than one line, and time that starts at 0.

| Figure | Content |
|---|---|
| 1 | Stationary angular rate, x, y, z, in deg/s versus time |
| 2 | Stationary acceleration, x, y, z, in m/s<sup>2</sup> versus time |
| 3 | Stationary yaw, pitch, roll in degrees versus time, recovered from your quaternions |
| 4 | Histograms of the stationary noise, gyro x, y, z and accelerometer x, y, z, each with its mean removed and a Gaussian of the same standard deviation drawn over it |
| 5 | Allan deviation of gyro x, y, z from the long data set, log-log, with N, B, and K marked. `plot_allan` in Part 3 does the marking |
| 6 | Three clips of 1 to 5 s from your motion data set: (a) the turn, gyro z and yaw; (b) a tip, the three accelerometer axes trading gravity; (c) the taps, the acceleration spikes. Say what happened in each |

# Part 3: Allan variance

A gyro that sits perfectly still still reports a rate. Averaging helps, up to a point: average for a second and white noise falls, average for an hour and slow drifts take over. The Allan deviation measures the noise left after averaging for a time $\tau$, for every $\tau$ at once, and the shape of that curve tells you which kind of noise dominates where.

Integrate the rate samples $\omega_k$, taken every $\tau_0 = 1/f_s$, into an angle $\theta_k = \tau_0 \sum_{i<k} \omega_i$. For clusters of $m$ samples, $\tau = m\tau_0$, the overlapping Allan variance is

$$\sigma^2(\tau) = \frac{1}{2\tau^2 (L - 2m)} \sum_{k=1}^{L-2m} \left(\theta_{k+2m} - 2\theta_{k+m} + \theta_k\right)^2$$

where $L$ is the number of angle samples. The Allan deviation is $\sigma(\tau)$, in the same units as the rate.

On a log-log plot each noise source has its own slope, and each parameter is read at a fixed point:

| Noise | $\sigma(\tau)$ | Slope | Where to read it | SI unit |
|---|---|---|---|---|
| Angle random walk, N | $N/\sqrt{\tau}$ | -1/2 | the -1/2 line at $\tau$ = 1 s | rad/s/√Hz, the same as rad/√s |
| Bias instability, B | about $0.664\,B$ | 0 | the minimum, divided by $\sqrt{2\ln 2/\pi} \approx 0.664$ | rad/s |
| Rate random walk, K | $K\sqrt{\tau/3}$ | +1/2 | the +1/2 line at $\tau$ = 3 s | rad/s/√s |

Datasheets use other units. Convert with:

$$N\,[\text{deg}/\sqrt{\text{h}}] = N\,[\text{rad}/\sqrt{\text{s}}] \times \tfrac{180}{\pi} \times 60 \qquad B\,[\text{deg/h}] = B\,[\text{rad/s}] \times \tfrac{180}{\pi} \times 3600$$

A noise density in deg/s/√Hz is $N$ in deg/√s, so multiply it by 60 to get deg/√h.

The three functions below compute the curve, read the parameters, and plot both. They stop the curve where fewer than ten clusters are left, because beyond that point the estimate is too noisy to read.

In [ ]:
def allan_deviation(rate, fs, n_taus=100, min_clusters=10):
    """Overlapping Allan deviation of a rate signal sampled at fs Hz.

    Returns tau in seconds and the Allan deviation in the units of rate.
    """
    rate = np.asarray(rate, dtype=float)
    tau0 = 1.0 / fs
    theta = np.concatenate(([0.0], np.cumsum(rate))) * tau0
    n = theta.size
    m = np.unique(np.logspace(0, np.log10((n - 1) // min_clusters), n_taus).astype(int))
    adev = np.empty(m.size)
    for i, mi in enumerate(m):
        d = theta[2 * mi:] - 2.0 * theta[mi:-mi] + theta[:-2 * mi]
        adev[i] = np.sqrt(np.mean(d ** 2) / (2.0 * (mi * tau0) ** 2))
    return m * tau0, adev

In [ ]:
def local_slope(logt, loga, half=3):
    """Slope of log10(adev) against log10(tau), fitted over 2*half+1 neighboring points."""
    s = np.full(logt.size, np.nan)
    for i in range(logt.size):
        lo, hi = max(0, i - half), min(logt.size, i + half + 1)
        if hi - lo >= 3:
            s[i] = np.polyfit(logt[lo:hi], loga[lo:hi], 1)[0]
    return s


def noise_parameters(tau, adev, tol=0.15):
    """Angle random walk N, bias instability B, and rate random walk K, in SI units.

    N: the -1/2 line through the curve where its slope is near -1/2, at tau = 1 s
    B: the minimum of the curve divided by sqrt(2 ln 2 / pi)
    K: the +1/2 line through the curve where its slope is near +1/2, at tau = 3 s
    N or K is NaN when the curve has no stretch with that slope.
    """
    logt, loga = np.log10(tau), np.log10(adev)
    slope = local_slope(logt, loga)
    i_min = int(np.argmin(adev))
    idx = np.arange(tau.size)
    B = adev[i_min] / np.sqrt(2 * np.log(2) / np.pi)
    sel = (idx <= i_min) & (np.abs(slope + 0.5) < tol)
    N = 10 ** np.median(loga[sel] + 0.5 * logt[sel]) if sel.any() else np.nan
    sel = (idx >= i_min) & (np.abs(slope - 0.5) < tol)
    K = 10 ** np.median(loga[sel] - 0.5 * logt[sel] + 0.5 * np.log10(3)) if sel.any() else np.nan
    return {"N": float(N), "B": float(B), "K": float(K), "tau_B": float(tau[i_min])}

In [ ]:
def plot_allan(ax, tau, adev, p, label):
    """Allan deviation on log-log axes, with the lines N, B, and K are read from."""
    line, = ax.loglog(tau, adev, label=label)
    c = line.get_color()
    if np.isfinite(p["N"]):
        ax.loglog(tau, p["N"] / np.sqrt(tau), "--", color=c, lw=0.8)
        ax.plot(1.0, p["N"], "o", color=c)
    ax.axhline(0.664 * p["B"], color=c, ls=":", lw=0.8)
    ax.plot(p["tau_B"], 0.664 * p["B"], "s", color=c)
    if np.isfinite(p["K"]):
        ax.loglog(tau, p["K"] * np.sqrt(tau / 3), "-.", color=c, lw=0.8)
        ax.plot(3.0, p["K"], "^", color=c)
    ax.set_xlabel("averaging time tau (s)")
    ax.set_ylabel("Allan deviation (rad/s)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()

### Check the tools on a signal whose answer you know

Before you trust any analysis on real data, run it on data where you know the answer. `simulate_gyro` builds a gyro signal from the three noise sources with values you choose: white noise for N, flicker noise for B, and a random walk for K. The values below are close to a VN-100.

In [ ]:
def simulate_gyro(N, B, K, fs=40.0, hours=5.0, bias=0.0, seed=0):
    """Synthetic gyro rate in rad/s with known N (rad/sqrt(s)), B (rad/s), K (rad/s/sqrt(s))."""
    rng = np.random.default_rng(seed)
    n = int(round(hours * 3600 * fs))
    dt = 1.0 / fs
    white = N / np.sqrt(dt) * rng.standard_normal(n)
    walk = np.cumsum(K * np.sqrt(dt) * rng.standard_normal(n))
    spectrum = np.fft.rfft(rng.standard_normal(n))
    f = np.fft.rfftfreq(n, dt)
    f[0] = f[1]
    flicker = np.fft.irfft(spectrum / np.sqrt(f), n) * B / np.sqrt(2 * np.pi * dt)
    return bias + white + flicker + walk

In [ ]:
truth = {"N": 6.0e-5, "B": 2.4e-5, "K": 1.0e-6}
fs_demo = 10.0
rate = simulate_gyro(**truth, fs=fs_demo, hours=24, bias=2e-3, seed=2)
tau, adev = allan_deviation(rate, fs_demo)
found = noise_parameters(tau, adev)

for k in ("N", "B", "K"):
    print(f"{k}: true {truth[k]:.2e}   found {found[k]:.2e}   ratio {found[k] / truth[k]:.2f}")
print(f"minimum of the curve at tau = {found['tau_B']:.0f} s")

fig, ax = plt.subplots(figsize=(7, 5))
plot_allan(ax, tau, adev, found, "synthetic gyro")
ax.set_title("24 h of synthetic gyro at 10 Hz, N, B, K known")
plt.show()

N comes back within a few percent. B comes back about 10 percent high, because angle random walk and rate random walk still add a little at the minimum. That is a property of the method, not a bug, and it is why datasheets say "in-run bias stability" rather than claiming an exact value. K is the least certain of the three: it lives at the long end of the curve, where there are few clusters. Change `seed` to 1, 3, and 0 and watch K move. With seed 0 it is not resolved at all.

Then try `hours=5` and `fs_demo = 40.0`, which is what your long data set looks like. K is often not resolved at all from five hours. When that happens with your data, say so in your README rather than reporting a number.

### Your long data set

`load_raw_lines` reads the raw sentences from the long data set, whether you have it as a rosbag folder or as a text file. Then parse every line with **your own driver's validation**. That is the point: the damaged lines in it are real, and your parser decides what survives.

Your parser should take one line and return `None` for a line it rejects. Otherwise it can return a dict with a `"gyro"` entry or the 12 numbers in sentence order. If your driver keeps its parser in a function at module level, you can import it directly, because your workspace is sourced.

In [ ]:
def load_raw_lines(path, topic=None):
    """Raw sentences from a .txt or .txt.gz file, or from a rosbag2 folder of strings."""
    path = Path(path).expanduser()
    if path.is_dir() and not (path / "metadata.yaml").exists():
        found = sorted(path.glob("*/metadata.yaml")) or sorted(path.glob("*.txt*"))
        if not found:
            raise FileNotFoundError(f"no rosbag or text file in {path}")
        path = found[0].parent if found[0].name == "metadata.yaml" else found[0]
    if path.is_file():
        opener = gzip.open if path.suffix == ".gz" else open
        with opener(path, "rt", encoding="ascii", errors="replace") as f:
            return [line.strip() for line in f if line.strip()]

    import yaml
    import rosbag2_py
    from rclpy.serialization import deserialize_message
    from rosidl_runtime_py.utilities import get_message

    meta = yaml.safe_load((path / "metadata.yaml").read_text())["rosbag2_bagfile_information"]
    reader = rosbag2_py.SequentialReader()
    reader.open(rosbag2_py.StorageOptions(uri=str(path), storage_id=meta["storage_identifier"]),
                rosbag2_py.ConverterOptions("cdr", "cdr"))
    types = {t.name: t.type for t in reader.get_all_topics_and_types()}
    if topic is None:
        strings = [name for name, t in types.items() if t == "std_msgs/msg/String"]
        if len(strings) != 1:
            raise ValueError(f"set LONG_TOPIC to one of {sorted(types)}")
        topic = strings[0]
    msg_type = get_message(types[topic])
    reader.set_filter(rosbag2_py.StorageFilter(topics=[topic]))
    lines = []
    while reader.has_next():
        _, data, _ = reader.read_next()
        m = deserialize_message(data, msg_type)
        lines.append(str(getattr(m, "data", "") or getattr(m, "vnymr_read", "")).strip())
    return lines

In [ ]:
LONG_DATA = Path("~/lab3_long_data").expanduser()   # the folder or file you downloaded
LONG_TOPIC = None   # the topic with the raw sentences, if the bag has more than one
FS = 40.0           # Hz, the rate the long data set was recorded at (Step 8)

# Use the parser from your own driver, for example:
#     from vn_driver.driver import parse_vnymr
def parse_vnymr(line):
    raise NotImplementedError("Import or paste the parser from your driver here")


def gyro_of(p):
    return p["gyro"] if isinstance(p, dict) else p[9:12]


long_params = None
if LONG_DATA.exists():
    lines = load_raw_lines(LONG_DATA, LONG_TOPIC)
    parsed = [parse_vnymr(s) for s in lines]
    gyro_long = np.array([gyro_of(p) for p in parsed if p is not None], dtype=float)
    rejected = len(lines) - len(gyro_long)
    print(f"{len(lines)} lines, {rejected} rejected ({100 * rejected / len(lines):.3f} percent), "
          f"{len(gyro_long) / FS / 3600:.2f} h of data")

    long_params = {}
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, axis in enumerate("xyz"):
        tau, adev = allan_deviation(gyro_long[:, i], FS)
        long_params[axis] = noise_parameters(tau, adev)
        plot_allan(ax, tau, adev, long_params[axis], f"gyro {axis}")
    ax.set_title("Figure 5: Allan deviation of the long data set")
    plt.show()
    for axis, p in long_params.items():
        print(f"{axis}: N {p['N']:.3e} rad/s/sqrt(Hz)   B {p['B']:.3e} rad/s   "
              f"K {p['K']:.3e} rad/s/sqrt(s)   minimum at {p['tau_B']:.0f} s")
else:
    print(f"Download the long data set to {LONG_DATA} first (Step 8).")

Each rejected line is a missing sample, and the code above simply closes the gap. A few dozen in five hours make no visible difference to the curve. If your parser rejects more than about 1 percent of the lines, look at what it rejects before you trust the result.

### Save your results

`save_noise_parameters` writes `noise_parameters.json` in SI units. Commit it in `Lab3/analysis/`. The grader compares it with the course's own analysis of the same data.

In [ ]:
def save_noise_parameters(params, path="noise_parameters.json", data_set=""):
    """Write N, B, K for each axis, in SI units, to a JSON file."""
    out = {"data_set": data_set, "sample_rate_hz": FS,
           "units": {"N": "rad/s/sqrt(Hz)", "B": "rad/s", "K": "rad/s/sqrt(s)", "tau_B": "s"}}
    for axis, p in params.items():
        out[axis] = {k: (float(v) if np.isfinite(v) else None) for k, v in p.items()}
    Path(path).write_text(json.dumps(out, indent=2) + "\n")
    print(f"wrote {path}")


if long_params is not None:
    save_noise_parameters(long_params, data_set=LONG_DATA.name)

In your `Lab3/README.md`, report the same numbers as a table with one row per axis. Give N, B, and K in SI units, add N in deg/√h and B in deg/h, which you convert yourself, and give the $\tau$ where the minimum fell. Write "not resolved" for K if the curve never rose at +1/2. The worked example shows the format.

# Analysis questions

Answer in your `Lab3/README.md`, with numbers from your own data. Each answer should be a short paragraph, not an essay.

**Question 1. Gravity, the magnetic field, and frames.** From your stationary data: which accelerometer axis reads about 9.8 m/s<sup>2</sup>, and with what sign? What does the sign tell you about which way the VN-100's z axis points, and how does that compare with REP 103? Then compute the magnitude of the magnetic field in microtesla, and the angle between the field and the horizontal plane, using the accelerometer to find down. Compare both with NOAA's model for your location (https://www.ngdc.noaa.gov/geomag/calculators/magcalc.shtml), and give one reason an indoor reading can differ.

**Question 2. Noise.** Give the mean and standard deviation of every gyro and accelerometer axis from your stationary data, as a table. Do your histograms look Gaussian? Point to one feature of your data that is not white Gaussian noise, such as a step at the printed resolution, a drift, or a periodic disturbance, and suggest a cause.

**Question 3. Noise parameters.** Report N, B, and K for each gyro axis from the long data set, and the $\tau$ where the minimum fell. Compare N and B with the VN-100 datasheet: a noise density of 0.0035 deg/s/√Hz and an in-run bias stability below 10 deg/h for the current hardware. Check the datasheet for your unit's revision. Which axis is worst? The rates in `$VNYMR` are corrected by the onboard filter's bias estimate. How would that correction change the long $\tau$ end of the curve, compared with the raw gyro?

**Question 4. Why five hours.** Compute the Allan deviation of your own five-minute recording for one gyro axis, and overlay it on the long data set's curve for the same axis. Over what range of $\tau$ do they agree? The relative error of an Allan deviation estimate is about $1/\sqrt{2(L/m - 1)}$ for $L$ samples and clusters of $m$. Use it to explain why five minutes cannot give you B or K.

**Question 5. From turning rate to heading.** In your turn clip, integrate gyro z over the turn and compare the angle with the change in the VN-100's own yaw. Explain the sign. Then, with your N and B for the z axis, estimate the heading error a robot would build up by integrating this gyro alone for 1 minute and for 1 hour: angle random walk grows as $N\sqrt{t}$, and a bias of size B grows as $Bt$. Which term dominates at each time scale, and what does that mean for dead reckoning?

# What to submit

**On Canvas:** one PDF with Figures 1 to 6, with your repository URL on the first page.

**In your repository:**

```
EECE5554/Lab3/
├── [NortheasternUsername]_lab3.txt   written by verify_lab3.sh, Section 9 completed
├── README.md                         noise parameter table and the five answers
├── analysis/
│   ├── your notebook or scripts
│   └── noise_parameters.json         written by Part 3
├── data/
│   ├── stationary/                   your rosbag, 5 minutes or more
│   └── motion/                       your team's rosbag
└── ros2_ws/src/
    ├── vn_msgs/
    └── vn_driver/
```

Do not commit `build/`, `install/`, or `log/`, the long data set, or videos. A 5 minute bag at 40 Hz is about 10 MB, which is fine.

# Assessment (100 points)

The driver and data sets are worth 40 points, and their points come from your self-check report, `[NortheasternUsername]_lab3.txt`, checked against what you committed. The analysis is worth 60. Partial credit is available within each item below, and the Canvas rubric shows the exact ratings.

**Driver (31 pts), from the self-check report**
<ul>
    <li> Packages, build, and message definition: vn_msgs with Vectornav.msg, vn_driver with driver.py and driver.launch.py, the workspace builds, and Vectornav has the four required fields (4 pts) </li>
    <li> Launch file takes the port argument (3 pts) </li>
    <li> Output rate set by writing register 7: the emulator confirms the write (3 pts), and messages arrive at 40 Hz with no repeats (3 pts) </li>
    <li> Headers: imu1_frame in all three, system time with nanoseconds (2 pts) </li>
    <li> Gyro and accelerometer values and units (3 pts) </li>
    <li> Magnetometer in tesla (3 pts) </li>
    <li> Orientation quaternion over the rotation test vector (5 pts) </li>
    <li> vnymr_read holds the whole sentence (1 pt) </li>
    <li> Damaged input: keeps publishing (2 pts), and publishes none of it (2 pts) </li>
</ul>

**Data sets (9 pts)**
<ul>
    <li> Stationary rosbag, at least 5 minutes at 40 Hz on /imu (5 pts) </li>
    <li> Motion rosbag on /imu (3 pts) </li>
    <li> No build, install, or log directories committed (1 pt) </li>
</ul>

**Figures (30 pts)**
<ul>
    <li> Figures 1, 2, 3, and 4 (4 pts each) </li>
    <li> Figure 5, with N, B, and K marked for all three axes (6 pts) </li>
    <li> Figure 6, three clips (2 pts each) </li>
    <li> Presentation across all six: axis labels with units, legends, time from 0 (2 pts) </li>
</ul>

**Noise parameter table (10 pts)**
<ul>
    <li> N, B, and K for x, y, and z in SI units, with the tau of each minimum (4 pts) </li>
    <li> N in deg/√h and B in deg/h, converted correctly (4 pts) </li>
    <li> noise_parameters.json committed and matching the README (2 pts) </li>
</ul>

**Analysis questions (20 pts)**
<ul>
    <li> Five questions, 4 points each, answered with your own numbers </li>
</ul>

### If something is not working

Push the self-check report exactly as generated, failures included, and use Section 9 to paste the exact error and your plan. An honest, specific failure report earns substantial partial credit. A hand-edited report claiming results the log does not show earns zero and is treated as an academic integrity matter. The grader cross-checks the report against what is actually committed, and may rerun the self-check on your workspace.

The driver is 31 of 100 points. If one check still fails, record your data anyway and earn the analysis marks.

# Appendix A: Common problems

**`ros2 topic hz /imu` settles at 10 Hz**
The rate change did not take effect. Look at the emulator's terminal. No `received` line means your command never arrived: you did not write it, wrote it before opening the port, or left off `\r\n`. `ERROR 03` means a wrong checksum. `ERROR 07` means a bad value, often a stray space.

**`TypeError: unicode strings are not supported, please encode to bytes`**
`ser.write` needs bytes: `ser.write(command.encode("ascii"))`.

**`UnicodeDecodeError`**
Real serial data contains bytes that are not text. Decode with `errors="replace"`.

**`ValueError: could not convert string to float`**
You converted before validating, or a field was empty. Check the header, the checksum, and the field count first, and catch `ValueError` around the conversion.

**The driver publishes nothing, but the emulator is streaming**
Check that you strip `\r\n` before checking the checksum, and that you compare against `$VNYMR,` exactly.

**The self-check fails orientation, but the numbers look reasonable**
It prints what your quaternion matches instead: degrees used as radians, yaw and roll swapped, the wrong rotation order, or components stored as w, x, y, z.

**Magnetic field values around 0.2 to 0.5**
Those are gauss. Multiply by 1e-4.

**`Permission denied` on `/dev/ttyUSB0`**
You are not in `dialout`, or you have not logged out since joining it. Check with `id -nG`. Do not use `chmod 777`.

**`Device or resource busy`**
minicom or another driver still has the port open. Close it.

**`ros2 launch` cannot find `driver.launch.py`**
`setup.py` is missing the `data_files` entry that installs `launch/`. Rebuild after fixing it.

**`ImportError: cannot import name 'Vectornav'`**
vn_msgs did not build, or you did not source `install/setup.bash` in this terminal. `ros2 interface show vn_msgs/msg/Vectornav` tells you which.

**`ros2 bag record` made an empty bag**
Your driver was not publishing yet. Confirm with `ros2 topic hz /imu` before you start recording.

**A bag has no `metadata.yaml` after a crash**
Rebuild it with `ros2 bag reindex data/stationary`.

**The notebook cannot import `rosbag2_py`**
Start VS Code from a terminal where ROS 2 and your workspace are sourced, and use the system Python 3.12 as the kernel.

If you are stuck for more than 30 minutes, post on `Piazza` with the command you ran and the full output. Paste text, not a screenshot of a terminal.

# Academic integrity

Teams share the VN-100 units and the motion recording. That is expected and permitted. Everything else is individual work: your driver, your stationary recording, your analysis code, your figures, and your answers. Identical figures or answers across submissions will be treated as a violation.